#### Fase 2.1 - Registrar el modelo ganador en Unity Catalog Model Registry

In [ ]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

EXPERIMENT_NAME = "/Shared/nyctaxi_fare_prediction"  # debe matchear el del training notebook
MODEL_NAME = "nyc_taxi_analytics.fare_prediction.fare_model"
MAE_IMPROVEMENT_THRESOLD = 10

#### Buscar el run ganador mas reciente

In [0]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="tags.mlflow.runName = 'xgboost_default'",
    order_by=["start_time DESC"],
    max_results=1
)

if runs.empty:
    raise ValueError(f"No se encontro ningun run 'xgboost_default' en el experimento {EXPERIMENT_NAME}. ")

xgb_run_id = runs.iloc[0]["run_id"]
mae_improvement = runs.iloc[0]["metrics.mae_improvement_pct"]

print(f"run_id encontrado: {xgb_run_id}")
print(f"Mejora de MAE vs naive: {mae_improvement:.1f}%")  

#### Registrar y apuntar el alias `champion`

In [ ]:
if mae_improvement > MAE_IMPROVEMENT_THRESOLD:
    model_uri = f"runs:/{xgb_run_id}/model"
    registered_model = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
    
    client = mlflow.tracking.MlflowClient()
    client.set_registered_model_alias(
        name=MODEL_NAME,
        alias="champion",
        version=registered_model.version,
    )

    print(f"✅Registrado {MODEL_NAME} version {registered_model.version} con alias 'champion'.")
else:
    print(f"⚠️ No se registra: mejora de MAE ({mae_improvement:.1f}%) por debajo "
          f"del umbral ({MAE_IMPROVEMENT_THRESOLD}%)")